In [1]:
from methylseg import MethylSegPathway, MethylDataPrep, HMMType
from pathlib import Path
import time
import yaml
from utils.methyl_tool_comparator import SharedPrepManager

In [2]:
def timeit(func, name, out_dir=Path("out")):
    out_file = Path(f"{out_dir}/{name}_execution_time.tsv")
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"Execution time for {func.__name__}: {end_time - start_time:.2f} seconds")
        with open(out_file, "w") as f:
            f.write(f"name\texecution_time\n")
            f.write(f"{func.__name__}\t{end_time - start_time:.2f}\n")
        return result
    return wrapper

In [5]:
def run_methylseg_test(sample_config, out_dir=Path("out")):
    sample_config = str(sample_config).strip()
    if not sample_config:
        return
    with open(sample_config) as handle:
        config = yaml.safe_load(handle)
    print(sample_config, config)
    sample_out_dir = out_dir / config["sample"]
    sample_out_dir.mkdir(parents=True, exist_ok=True)
    shared_prep_manager = SharedPrepManager(
        config["sample"],
        config["meth_file"],
        config["genome"],
        out_dir=out_dir,
        skip_450k=True
    )
    shared_outputs = shared_prep_manager.prepare()

    wgbs_test_sample_info, wgbs_test_sample_info_removed = MethylDataPrep(
        meth_file=shared_outputs.wgbs_tsv,
        sample_id=config["sample"],
        resolution="wgbs",
        remove_low_coverage_like_cpgs=True,
    ).prepare()
    
    meth_seg_pathway_sticky = MethylSegPathway(
        train_sample_info=wgbs_test_sample_info,
        hmm_type=HMMType.STICKY,
        out_dir=sample_out_dir / "sticky"
    )

    meth_seg_pathway_ct = MethylSegPathway(
        train_sample_info=wgbs_test_sample_info,
        hmm_type=HMMType.CT,
        out_dir=sample_out_dir / "ct"
    )
    print(f"Running methylseg test for sample {config['sample']} with meth file {config['meth_file']} and genome {config['genome']}")

    ct_outputs = timeit(
        meth_seg_pathway_ct.run_pathway,
        f"{config['sample']}_continuous_time_hmm",
        out_dir=sample_out_dir,
    )()
    sticky_outputs = timeit(
        meth_seg_pathway_sticky.run_pathway,
        f"{config['sample']}_sticky_hmm",
        out_dir=sample_out_dir,
    )()
    return {
        "continuous_time_hmm": ct_outputs,
        "sticky_hmm": sticky_outputs,
    }



In [6]:
for config in open("/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/analysis/01_region_calling_analysis/slurm_code/configs.txt"):
    run_methylseg_test(config)

/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/analysis/01_region_calling_analysis/slurm_code/configs/SRR26107673_config.yaml {'genome': 'hg19', 'meth_file': '/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/data/methylation_data/SRR26107673.beta', 'sample': 'SRR26107673'}
[2026-08-05 11:18:36] Building shared prep artifacts in out/SRR26107673/shared_prep
Running methylseg test for sample SRR26107673 with meth file /uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/data/methylation_data/SRR26107673.beta and genome hg19
/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/analysis/01_region_calling_analysis/slurm_code/configs/WGBS_colon-primary-normal_1_meth_config.yaml {'genome': 'hg38', 'meth_file': '/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/data/methylation_data/WGBS_colon-primary-normal_1_meth.bed.gz', 'sample': 'WGBS_colon-primary-normal_1_m

KeyboardInterrupt: 